# M5.3 — Logistic Regression baseline

Milestone:

`M5 — Modeling baseline`

Substep:

`M5.3 — Logistic Regression baseline`

Official runs:

`M5-LR-LONG-B01`

`M5-LR-SHORT-B01`

Câu hỏi:

> Với Baseline Feature/Preprocessing Contract đã khóa ở M4 và không can thiệp class imbalance, Logistic Regression tạo được baseline modeling evidence nào trên W_LONG và W_SHORT?

Notebook này:

- khóa exact Logistic Regression baseline config **trước khi đọc validation result**;
- dùng cùng config cho W_LONG và W_SHORT;
- dùng `IMBALANCE_STRATEGY = NONE`;
- dùng shared metric/runner semantics từ M5.2;
- giữ probability/risk score;
- dùng default model decision rule làm baseline comparison rule;
- không tuning;
- không temporal CV;
- không chọn final threshold;
- không dùng FINAL TEST;
- không tự chọn W_LONG/W_SHORT winner.

M5.3 chỉ được khóa sau khi notebook chạy thật và runtime output được review.

## 1. Contract kế thừa

M5.3 kế thừa:

- `CANON-M5.1 — Modeling Charter`;
- `M5.2 — Artifact Handoff / Metric Contract / Shared Runner`;
- M4 baseline feature/preprocessing contract;
- M3 evaluation và training-window comparison protocol.

Các invariant:

```text
Input:
canonical M4.7 persisted matrices

Feature schema:
47-column CSR float32

Positive class:
fraud = 1

Model family:
Logistic Regression

Training windows:
W_LONG
W_SHORT

Validation:
same external VALIDATION

Imbalance:
NONE

Primary:
F1_fraud

Secondary:
Recall_fraud
Precision_fraud

Threshold policy:
DEFAULT_MODEL_DECISION_RULE
not final threshold

FINAL TEST:
NO ACCESS
```

## 2. Exact Logistic Regression baseline configuration

M5.1 để exact Logistic Regression config ở trạng thái OPEN cho M5.3.

M5.3 khóa baseline config **trước runtime model score**:

```text
Model:
sklearn.linear_model.LogisticRegression

penalty:
l2

C:
1.0

solver:
saga

max_iter:
200

tol:
1e-4

fit_intercept:
True

class_weight:
None

random_state:
42
```

### Lý do

`solver="saga"` được chọn vì baseline matrix là CSR sparse và có hàng triệu rows.

L2 + `C=1.0` là một cấu hình baseline đơn giản, không phải kết quả search/tuning.

`class_weight=None` giữ đúng no-intervention baseline.

`random_state=42` phục vụ reproducibility.

`max_iter=200` là iteration budget kỹ thuật. Nếu xuất hiện `ConvergenceWarning`, run không đủ điều kiện để diễn giải; cần xử lý convergence và chạy lại **cả controlled pair** bằng cùng config.

Không được thử nhiều C/solver rồi chọn validation score cao nhất trong M5.3.

## 3. Runtime sequence

```text
environment
        ↓
locate M4.7 + M5.2 contracts
        ↓
load W_LONG / W_SHORT / VALIDATION
        ↓
input integrity gate
        ↓
define shared M5.2 metric/runner semantics
        ↓
freeze LR-B01 config to disk
        ↓
run LR-LONG
        ↓
run LR-SHORT
        ↓
warning / convergence gate
        ↓
probability + default-decision-rule gate
        ↓
controlled-pair comparability gate
        ↓
persist predictions / risk scores / summaries
        ↓
round-trip gate
        ↓
M5.3 technical gate
        ↓
runtime review with AI
```

Nếu một gate fail:

`STOP — không diễn giải metric`.

In [1]:

from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Optional
import hashlib
import json
import platform
import sys
import time
import warnings

import numpy as np

try:
    from scipy import sparse
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Thiếu scipy. Chạy `%pip install scipy scikit-learn`, "
        "Restart Kernel rồi Run All."
    ) from exc

try:
    import sklearn
    from sklearn.base import clone
    from sklearn.exceptions import ConvergenceWarning
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import (
        accuracy_score,
        confusion_matrix,
        f1_score,
        precision_score,
        recall_score,
    )
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Thiếu scikit-learn. Chạy `%pip install scipy scikit-learn`, "
        "Restart Kernel rồi Run All."
    ) from exc


print("Python:")
print(sys.version)

print("\nExecutable:")
print(sys.executable)

print("\nPlatform:")
print(platform.platform())

print("\nNumPy:")
print(np.__version__)

print("\nscikit-learn:")
print(sklearn.__version__)


Python:
3.14.6 (main, Jun 10 2026, 10:03:53) [Clang 21.0.0 (clang-2100.0.123.102)]

Executable:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/.venv/bin/python

Platform:
macOS-26.6.2-arm64-arm-64bit-Mach-O

NumPy:
2.5.3

scikit-learn:
1.9.1


## 4. Locate canonical M4.7 artifacts và M5.2 contract

In [2]:

M4_ARTIFACT_RELATIVE_DIR = (
    Path("data")
    / "processed"
    / "m4_07_baseline_ready"
)

M5_02_CONTRACT_RELATIVE_PATH = (
    Path("data")
    / "processed"
    / "m5_02_modeling_contract"
    / "m5_02_modeling_contract.json"
)

M5_03_OUTPUT_RELATIVE_DIR = (
    Path("data")
    / "processed"
    / "m5_03_logistic_regression_baseline"
)


candidate_roots = [
    Path.cwd(),
    *list(Path.cwd().parents)[:5],
]


PROJECT_ROOT = None

for candidate in candidate_roots:
    candidate = candidate.resolve()

    if (
        (
            candidate
            / M4_ARTIFACT_RELATIVE_DIR
            / "manifest.json"
        ).exists()
        and
        (
            candidate
            / M5_02_CONTRACT_RELATIVE_PATH
        ).exists()
    ):
        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy PROJECT_ROOT chứa cả "
        "M4.7 manifest và M5.2 modeling contract."
    )


ARTIFACT_DIR = (
    PROJECT_ROOT
    / M4_ARTIFACT_RELATIVE_DIR
)

M5_02_CONTRACT_PATH = (
    PROJECT_ROOT
    / M5_02_CONTRACT_RELATIVE_PATH
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / M5_03_OUTPUT_RELATIVE_DIR
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nARTIFACT_DIR:")
print(ARTIFACT_DIR)

print("\nM5_02_CONTRACT_PATH:")
print(M5_02_CONTRACT_PATH)

print("\nOUTPUT_DIR:")
print(OUTPUT_DIR)


PROJECT_ROOT:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening

ARTIFACT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m4_07_baseline_ready

M5_02_CONTRACT_PATH:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m5_02_modeling_contract/m5_02_modeling_contract.json

OUTPUT_DIR:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m5_03_logistic_regression_baseline


## 5. Canonical expected signatures

In [3]:

EXPECTED_OUTPUT_WIDTH = 47

EXPECTED_W_LONG_ROWS = 6_855_270
EXPECTED_W_SHORT_ROWS = 1_721_615
EXPECTED_VALIDATION_ROWS = 712_458

EXPECTED_W_LONG_FRAUD = 9_606
EXPECTED_W_SHORT_FRAUD = 2_491
EXPECTED_VALIDATION_FRAUD = 1_052

EXPECTED_NNZ = {
    "X_train_w_long": 61_900_193,
    "X_train_w_short": 15_542_381,
    "X_validation_w_long": 6_430_339,
    "X_validation_w_short": 6_430_339,
}

M4_PIPELINE_VERSION = "M4.7-baseline-v1"
M5_RUNNER_VERSION = "M5.2-shared-runner-v1"

IMBALANCE_STRATEGY = "NONE"
DEFAULT_RANDOM_STATE = 42
BASELINE_THRESHOLD_POLICY = "DEFAULT_MODEL_DECISION_RULE"

FEATURE_VERSION = "Feature Specification v1.0"
PREPROCESSING_VERSION = "Preprocessing Specification v1.0"
MATRIX_SCHEMA_VERSION = "Baseline Matrix Schema v1.0"

LR_MODEL_FAMILY = "Logistic Regression"
LR_MODEL_ID = "SKLEARN_LOGISTIC_REGRESSION"
LR_CONFIG_ID = "LR-B01-SAGA-L2-C1"

LR_LONG_EXPERIMENT_ID = "M5-LR-LONG-B01"
LR_SHORT_EXPERIMENT_ID = "M5-LR-SHORT-B01"

print("M5 runner version:")
print(M5_RUNNER_VERSION)

print("\nLR config ID:")
print(LR_CONFIG_ID)

print("\nImbalance strategy:")
print(IMBALANCE_STRATEGY)

print("\nThreshold policy:")
print(BASELINE_THRESHOLD_POLICY)


M5 runner version:
M5.2-shared-runner-v1

LR config ID:
LR-B01-SAGA-L2-C1

Imbalance strategy:
NONE

Threshold policy:
DEFAULT_MODEL_DECISION_RULE


## 6. Load manifest, feature names và M5.2 contract

In [4]:

with open(
    ARTIFACT_DIR / "manifest.json",
    "r",
    encoding="utf-8",
) as file:
    manifest = json.load(file)


with open(
    ARTIFACT_DIR / "feature_names.json",
    "r",
    encoding="utf-8",
) as file:
    feature_names = json.load(file)


with open(
    M5_02_CONTRACT_PATH,
    "r",
    encoding="utf-8",
) as file:
    m5_02_contract = json.load(file)


assert manifest["pipeline_version"] == M4_PIPELINE_VERSION
assert manifest["output_width"] == EXPECTED_OUTPUT_WIDTH
assert manifest["matrix_dtype"] == "float32"
assert str(manifest["matrix_format"]).upper() == "CSR"
assert manifest["target_dtype"] == "int8"
assert manifest["raw_identifiers_in_X"] is False
assert manifest["target_in_X"] is False
assert manifest["final_test_used"] is False

assert len(feature_names) == EXPECTED_OUTPUT_WIDTH
assert len(set(feature_names)) == EXPECTED_OUTPUT_WIDTH

assert m5_02_contract["runner_version"] == M5_RUNNER_VERSION
assert m5_02_contract["input_pipeline_version"] == M4_PIPELINE_VERSION
assert m5_02_contract["feature_count"] == EXPECTED_OUTPUT_WIDTH
assert m5_02_contract["matrix_format"] == "CSR"
assert m5_02_contract["matrix_dtype"] == "float32"
assert m5_02_contract["target_dtype"] == "int8"
assert m5_02_contract["imbalance_strategy"] == IMBALANCE_STRATEGY
assert m5_02_contract["random_state_default"] == DEFAULT_RANDOM_STATE
assert (
    m5_02_contract["threshold_policy"]
    == BASELINE_THRESHOLD_POLICY
)
assert m5_02_contract["final_test_access_allowed"] is False
assert m5_02_contract["experiment_schema_covers_m5_1"] is True


print("M4 pipeline version:")
print(manifest["pipeline_version"])

print("\nM5.2 runner version:")
print(m5_02_contract["runner_version"])

print("\nFeature count:")
print(len(feature_names))

print("\nM5.3 MANIFEST / CONTRACT GATE: PASS")


M4 pipeline version:
M4.7-baseline-v1

M5.2 runner version:
M5.2-shared-runner-v1

Feature count:
47

M5.3 MANIFEST / CONTRACT GATE: PASS


## 7. Load modeling matrices và targets

Không load lineage vào classifier.

Không có FINAL TEST path trong notebook.

In [5]:

load_start = time.perf_counter()

X_train_w_long = sparse.load_npz(
    ARTIFACT_DIR / "X_train_w_long.npz"
)

X_train_w_short = sparse.load_npz(
    ARTIFACT_DIR / "X_train_w_short.npz"
)

X_validation_w_long = sparse.load_npz(
    ARTIFACT_DIR / "X_validation_w_long.npz"
)

X_validation_w_short = sparse.load_npz(
    ARTIFACT_DIR / "X_validation_w_short.npz"
)

y_train_w_long = np.load(
    ARTIFACT_DIR / "y_train_w_long.npy",
    allow_pickle=False,
)

y_train_w_short = np.load(
    ARTIFACT_DIR / "y_train_w_short.npy",
    allow_pickle=False,
)

y_validation = np.load(
    ARTIFACT_DIR / "y_validation.npy",
    allow_pickle=False,
)

load_seconds = time.perf_counter() - load_start

print("Artifact load seconds:")
print(round(load_seconds, 2))

print("\nM5.3 ARTIFACT LOAD: COMPLETE")


Artifact load seconds:
0.74

M5.3 ARTIFACT LOAD: COMPLETE


## 8. Input integrity preflight

M5.3 rechecks only the signatures needed before expensive model fit.

In [6]:

modeling_matrices = {
    "X_train_w_long": X_train_w_long,
    "X_train_w_short": X_train_w_short,
    "X_validation_w_long": X_validation_w_long,
    "X_validation_w_short": X_validation_w_short,
}

expected_shapes = {
    "X_train_w_long": (
        EXPECTED_W_LONG_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_train_w_short": (
        EXPECTED_W_SHORT_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_validation_w_long": (
        EXPECTED_VALIDATION_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
    "X_validation_w_short": (
        EXPECTED_VALIDATION_ROWS,
        EXPECTED_OUTPUT_WIDTH,
    ),
}


for name, matrix in modeling_matrices.items():
    assert sparse.isspmatrix_csr(matrix)
    assert matrix.dtype == np.float32
    assert matrix.shape == expected_shapes[name]
    assert matrix.nnz == EXPECTED_NNZ[name]
    assert np.isfinite(matrix.data).all()


assert y_train_w_long.dtype == np.int8
assert y_train_w_short.dtype == np.int8
assert y_validation.dtype == np.int8

assert len(y_train_w_long) == EXPECTED_W_LONG_ROWS
assert len(y_train_w_short) == EXPECTED_W_SHORT_ROWS
assert len(y_validation) == EXPECTED_VALIDATION_ROWS

assert int(y_train_w_long.sum()) == EXPECTED_W_LONG_FRAUD
assert int(y_train_w_short.sum()) == EXPECTED_W_SHORT_FRAUD
assert int(y_validation.sum()) == EXPECTED_VALIDATION_FRAUD

assert set(np.unique(y_train_w_long).tolist()) <= {0, 1}
assert set(np.unique(y_train_w_short).tolist()) <= {0, 1}
assert set(np.unique(y_validation).tolist()) <= {0, 1}


print("W_LONG:")
print(
    X_train_w_long.shape,
    X_train_w_long.dtype,
    "fraud=",
    int(y_train_w_long.sum()),
)

print("\nW_SHORT:")
print(
    X_train_w_short.shape,
    X_train_w_short.dtype,
    "fraud=",
    int(y_train_w_short.sum()),
)

print("\nVALIDATION:")
print(
    X_validation_w_long.shape,
    "fraud=",
    int(y_validation.sum()),
)

print("\nM5.3 INPUT INTEGRITY GATE: PASS")


W_LONG:
(6855270, 47) float32 fraud= 9606

W_SHORT:
(1721615, 47) float32 fraud= 2491

VALIDATION:
(712458, 47) fraud= 1052

M5.3 INPUT INTEGRITY GATE: PASS


## 9. Shared canonical metric implementation

Implementation giữ cùng semantics với M5.2.

In [7]:

def compute_binary_metrics(
    y_true,
    y_pred,
):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    assert y_true.ndim == 1
    assert y_pred.ndim == 1
    assert len(y_true) == len(y_pred)

    assert set(np.unique(y_true).tolist()) <= {0, 1}
    assert set(np.unique(y_pred).tolist()) <= {0, 1}

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1],
    )

    tn, fp, fn, tp = cm.ravel()

    predicted_positive_count = int(tp + fp)

    predicted_positive_rate = (
        predicted_positive_count / len(y_true)
        if len(y_true)
        else np.nan
    )

    return {
        "f1_fraud": float(
            f1_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "recall_fraud": float(
            recall_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "precision_fraud": float(
            precision_score(
                y_true,
                y_pred,
                pos_label=1,
                zero_division=0,
            )
        ),
        "accuracy_reference": float(
            accuracy_score(
                y_true,
                y_pred,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "predicted_positive_count": predicted_positive_count,
        "predicted_positive_rate": float(predicted_positive_rate),
        "validation_rows": int(len(y_true)),
    }


## 10. Shared runner semantics từ M5.2

Runner:

- clone estimator;
- fit TRAIN;
- predict VALIDATION;
- giữ probability/decision score;
- capture warnings;
- tính canonical metrics;
- ghi experiment metadata;
- `final_test_accessed = false`.

In [8]:

def _json_safe_value(value):
    if isinstance(
        value,
        (str, int, float, bool, type(None)),
    ):
        return value

    if isinstance(value, np.generic):
        return value.item()

    if isinstance(value, (list, tuple)):
        return [
            _json_safe_value(item)
            for item
            in value
        ]

    if isinstance(value, dict):
        return {
            str(key): _json_safe_value(item)
            for key, item
            in value.items()
        }

    return repr(value)


def extract_positive_score(
    fitted_estimator,
    X,
):
    if hasattr(fitted_estimator, "predict_proba"):
        probabilities = fitted_estimator.predict_proba(X)
        classes = np.asarray(fitted_estimator.classes_)

        positive_positions = np.flatnonzero(
            classes == 1
        )

        if len(positive_positions) != 1:
            raise RuntimeError(
                "Không xác định được positive class = 1 "
                "trong predict_proba."
            )

        score = probabilities[
            :,
            positive_positions[0],
        ]

        return (
            np.asarray(
                score,
                dtype=np.float64,
            ),
            "predict_proba",
        )

    if hasattr(fitted_estimator, "decision_function"):
        score = np.asarray(
            fitted_estimator.decision_function(X),
            dtype=np.float64,
        )

        if score.ndim != 1:
            raise RuntimeError(
                "decision_function không phải binary 1-D score."
            )

        return score, "decision_function"

    return None, None


@dataclass
class ExperimentRunOutput:
    summary: Dict[str, Any]
    y_pred: np.ndarray
    risk_score: Optional[np.ndarray]
    risk_score_kind: Optional[str]
    warning_messages: List[str]


def run_baseline_experiment(
    *,
    experiment_id,
    estimator,
    model_family,
    model_id,
    model_config_id,
    X_train,
    y_train,
    X_validation,
    y_validation,
    training_window_id,
    random_state=None,
    feature_version=FEATURE_VERSION,
    preprocessing_version=PREPROCESSING_VERSION,
    matrix_schema_version=MATRIX_SCHEMA_VERSION,
    imbalance_strategy=IMBALANCE_STRATEGY,
    threshold_policy=BASELINE_THRESHOLD_POLICY,
    decision="OPEN — REQUIRES RUNTIME REVIEW",
    next_action="REVIEW RUN OUTPUT",
):
    assert imbalance_strategy == "NONE"
    assert training_window_id in {"W_LONG", "W_SHORT"}

    assert isinstance(model_family, str) and model_family.strip()
    assert isinstance(model_id, str) and model_id.strip()
    assert isinstance(model_config_id, str) and model_config_id.strip()

    if random_state is not None:
        assert isinstance(random_state, (int, np.integer))

    assert sparse.isspmatrix_csr(X_train)
    assert sparse.isspmatrix_csr(X_validation)

    assert X_train.shape[1] == EXPECTED_OUTPUT_WIDTH
    assert X_validation.shape[1] == EXPECTED_OUTPUT_WIDTH

    assert X_train.shape[0] == len(y_train)
    assert X_validation.shape[0] == len(y_validation)

    assert np.isfinite(X_train.data).all()
    assert np.isfinite(X_validation.data).all()

    assert set(np.unique(y_train).tolist()) <= {0, 1}
    assert set(np.unique(y_validation).tolist()) <= {0, 1}

    model = clone(estimator)

    with warnings.catch_warnings(record=True) as caught_warnings:
        warnings.simplefilter("always")

        fit_start = time.perf_counter()

        model.fit(
            X_train,
            y_train,
        )

        fit_seconds = (
            time.perf_counter()
            - fit_start
        )

        prediction_start = time.perf_counter()

        y_pred = model.predict(
            X_validation
        )

        risk_score, risk_score_kind = (
            extract_positive_score(
                model,
                X_validation,
            )
        )

        prediction_seconds = (
            time.perf_counter()
            - prediction_start
        )

    y_pred = np.asarray(
        y_pred,
        dtype=np.int8,
    )

    metrics = compute_binary_metrics(
        y_validation,
        y_pred,
    )

    warning_messages = [
        (
            f"{item.category.__name__}: "
            f"{item.message}"
        )
        for item
        in caught_warnings
    ]

    model_params = {
        key: _json_safe_value(value)
        for key, value
        in model.get_params(
            deep=False
        ).items()
    }

    summary = {
        "runner_version": M5_RUNNER_VERSION,
        "experiment_id": str(experiment_id),
        "run_status": "COMPLETED",

        "model_family": str(model_family),
        "model_id": str(model_id),
        "model_config_id": str(model_config_id),
        "model_class": model.__class__.__name__,
        "model_module": model.__class__.__module__,
        "model_params": model_params,

        "training_window_id": training_window_id,

        "feature_version": feature_version,
        "preprocessing_version": preprocessing_version,
        "matrix_schema_version": matrix_schema_version,
        "feature_count": int(X_train.shape[1]),

        "training_rows": int(X_train.shape[0]),
        "training_fraud_rows": int(np.asarray(y_train).sum()),
        "validation_rows": int(X_validation.shape[0]),
        "validation_fraud_rows": int(np.asarray(y_validation).sum()),

        "imbalance_strategy": imbalance_strategy,
        "random_state": (
            None
            if random_state is None
            else int(random_state)
        ),
        "threshold_policy": threshold_policy,

        "fit_seconds": float(fit_seconds),
        "prediction_seconds": float(prediction_seconds),

        "risk_score_available": risk_score is not None,
        "risk_score_kind": risk_score_kind,

        "warning_count": len(warning_messages),
        "warnings": warning_messages,
        "errors": [],

        "final_test_accessed": False,
        "integrity_gate_result": "PASS",

        "decision": str(decision),
        "next_action": str(next_action),

        **metrics,
    }

    return ExperimentRunOutput(
        summary=summary,
        y_pred=y_pred,
        risk_score=(
            None
            if risk_score is None
            else np.asarray(
                risk_score,
                dtype=np.float32,
            )
        ),
        risk_score_kind=risk_score_kind,
        warning_messages=warning_messages,
    )


def save_experiment_output(
    run_output,
    output_dir,
):
    output_dir = Path(output_dir)
    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    experiment_id = run_output.summary["experiment_id"]

    prediction_path = (
        output_dir
        / f"{experiment_id}__y_pred.npy"
    )

    summary_path = (
        output_dir
        / f"{experiment_id}__summary.json"
    )

    np.save(
        prediction_path,
        run_output.y_pred,
        allow_pickle=False,
    )

    risk_score_path = None

    if run_output.risk_score is not None:
        risk_score_path = (
            output_dir
            / f"{experiment_id}__risk_score.npy"
        )

        np.save(
            risk_score_path,
            run_output.risk_score,
            allow_pickle=False,
        )

    persisted_summary = dict(
        run_output.summary
    )

    persisted_summary["prediction_file"] = prediction_path.name
    persisted_summary["risk_score_file"] = (
        None
        if risk_score_path is None
        else risk_score_path.name
    )

    with open(
        summary_path,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            persisted_summary,
            file,
            ensure_ascii=False,
            indent=2,
        )

    return {
        "summary_path": summary_path,
        "prediction_path": prediction_path,
        "risk_score_path": risk_score_path,
    }


## 11. Define Logistic Regression B01

Config được tạo một lần rồi clone cho cả hai window.

In [9]:

LR_BASELINE_CONFIG = {
    "penalty": "l2",
    "C": 1.0,
    "solver": "saga",
    "max_iter": 200,
    "tol": 1e-4,
    "fit_intercept": True,
    "class_weight": None,
    "random_state": DEFAULT_RANDOM_STATE,
}


lr_estimator_template = LogisticRegression(
    **LR_BASELINE_CONFIG
)


actual_params = lr_estimator_template.get_params(
    deep=False
)


for key, expected_value in LR_BASELINE_CONFIG.items():
    assert actual_params[key] == expected_value


assert actual_params["class_weight"] is None
assert actual_params["solver"] == "saga"
assert actual_params["penalty"] == "l2"
assert actual_params["C"] == 1.0
assert actual_params["random_state"] == DEFAULT_RANDOM_STATE


print("Logistic Regression baseline config:")

for key in sorted(LR_BASELINE_CONFIG):
    print(
        f"{key}: "
        f"{LR_BASELINE_CONFIG[key]}"
    )

print("\nM5.3 LR CONFIG DEFINITION GATE: PASS")


Logistic Regression baseline config:
C: 1.0
class_weight: None
fit_intercept: True
max_iter: 200
penalty: l2
random_state: 42
solver: saga
tol: 0.0001

M5.3 LR CONFIG DEFINITION GATE: PASS


## 12. Freeze config trước model result

Cell này phải chạy **trước** LR-LONG/LR-SHORT.

Nó persist exact config + fingerprint để chứng minh controlled pair không được chọn config sau khi nhìn validation score.

In [10]:

config_lock_payload = {
    "m5_substep": "M5.3",
    "model_family": LR_MODEL_FAMILY,
    "model_id": LR_MODEL_ID,
    "model_config_id": LR_CONFIG_ID,
    "config": LR_BASELINE_CONFIG,
    "feature_version": FEATURE_VERSION,
    "preprocessing_version": PREPROCESSING_VERSION,
    "matrix_schema_version": MATRIX_SCHEMA_VERSION,
    "imbalance_strategy": IMBALANCE_STRATEGY,
    "random_state": DEFAULT_RANDOM_STATE,
    "threshold_policy": BASELINE_THRESHOLD_POLICY,
    "training_windows": [
        "W_LONG",
        "W_SHORT",
    ],
    "validation_role": "EXTERNAL_VALIDATION",
    "final_test_access": False,
    "systematic_tuning": False,
}


config_canonical_json = json.dumps(
    config_lock_payload,
    ensure_ascii=False,
    sort_keys=True,
    separators=(",", ":"),
)


config_fingerprint = hashlib.sha256(
    config_canonical_json.encode("utf-8")
).hexdigest()


config_lock_payload[
    "config_fingerprint_sha256"
] = config_fingerprint


CONFIG_LOCK_PATH = (
    OUTPUT_DIR
    / "m5_03_lr_baseline_config_lock.json"
)


with open(
    CONFIG_LOCK_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        config_lock_payload,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


assert CONFIG_LOCK_PATH.exists()
assert CONFIG_LOCK_PATH.stat().st_size > 0

with open(
    CONFIG_LOCK_PATH,
    "r",
    encoding="utf-8",
) as file:
    reloaded_config_lock = json.load(file)


assert (
    reloaded_config_lock[
        "config_fingerprint_sha256"
    ]
    == config_fingerprint
)

assert (
    reloaded_config_lock[
        "imbalance_strategy"
    ]
    == "NONE"
)

assert (
    reloaded_config_lock[
        "final_test_access"
    ]
    is False
)

assert (
    reloaded_config_lock[
        "systematic_tuning"
    ]
    is False
)


print("Config lock path:")
print(CONFIG_LOCK_PATH)

print("\nConfig SHA256:")
print(config_fingerprint)

print("\nM5.3 PRE-RESULT CONFIG LOCK GATE: PASS")


Config lock path:
/Users/minhtoan/SE/HocTrenLop/Trí tuệ nhân tạo/fraud-risk-screening/data/processed/m5_03_logistic_regression_baseline/m5_03_lr_baseline_config_lock.json

Config SHA256:
95f5c822212664243802e85f81b4297bb7129a14813d900ee1160ab8d689423f

M5.3 PRE-RESULT CONFIG LOCK GATE: PASS


## 13. Authorized modeling inputs

Không có FINAL TEST key.

In [11]:

MODELING_INPUTS = {
    "W_LONG": {
        "X_train": X_train_w_long,
        "y_train": y_train_w_long,
        "X_validation": X_validation_w_long,
        "y_validation": y_validation,
        "experiment_id": LR_LONG_EXPERIMENT_ID,
    },
    "W_SHORT": {
        "X_train": X_train_w_short,
        "y_train": y_train_w_short,
        "X_validation": X_validation_w_short,
        "y_validation": y_validation,
        "experiment_id": LR_SHORT_EXPERIMENT_ID,
    },
}


assert set(MODELING_INPUTS) == {
    "W_LONG",
    "W_SHORT",
}

assert "FINAL_TEST" not in MODELING_INPUTS


print("Authorized modeling inputs:")
print(sorted(MODELING_INPUTS))

print("\nM5.3 MODELING INPUT GATE: PASS")


Authorized modeling inputs:
['W_LONG', 'W_SHORT']

M5.3 MODELING INPUT GATE: PASS


# 14. Fit controlled Logistic Regression pair

Hai runs dùng cùng:

- estimator template;
- exact hyperparameters;
- feature/preprocessing versions;
- `IMBALANCE_STRATEGY = NONE`;
- random state;
- validation population;
- threshold policy;
- metric implementation.

Chỉ training window thay đổi.

Không sửa config giữa hai runs.

In [12]:

run_outputs = {}


pair_start = time.perf_counter()


for training_window_id in [
    "W_LONG",
    "W_SHORT",
]:
    item = MODELING_INPUTS[
        training_window_id
    ]

    print(
        "\n"
        + "=" * 72
    )

    print(
        "Starting:",
        item["experiment_id"],
    )

    print(
        "Training window:",
        training_window_id,
    )

    print(
        "Training rows:",
        f"{item['X_train'].shape[0]:,}",
    )

    print(
        "Training fraud:",
        f"{int(item['y_train'].sum()):,}",
    )

    run_output = run_baseline_experiment(
        experiment_id=item[
            "experiment_id"
        ],
        estimator=lr_estimator_template,
        model_family=LR_MODEL_FAMILY,
        model_id=LR_MODEL_ID,
        model_config_id=LR_CONFIG_ID,
        X_train=item["X_train"],
        y_train=item["y_train"],
        X_validation=item[
            "X_validation"
        ],
        y_validation=item[
            "y_validation"
        ],
        training_window_id=
            training_window_id,
        random_state=
            DEFAULT_RANDOM_STATE,
        decision=(
            "OPEN — REQUIRES M5.3 RUNTIME REVIEW"
        ),
        next_action=(
            "COMPARE AS BASELINE EVIDENCE; "
            "DO NOT SELECT FINAL WINNER IN M5.3"
        ),
    )

    run_outputs[
        training_window_id
    ] = run_output

    summary = run_output.summary

    print(
        "\nCompleted:",
        summary["experiment_id"],
    )

    print(
        "fit_seconds:",
        round(
            summary["fit_seconds"],
            3,
        ),
    )

    print(
        "prediction_seconds:",
        round(
            summary[
                "prediction_seconds"
            ],
            3,
        ),
    )

    print(
        "warning_count:",
        summary["warning_count"],
    )

    print(
        "f1_fraud:",
        summary["f1_fraud"],
    )

    print(
        "recall_fraud:",
        summary["recall_fraud"],
    )

    print(
        "precision_fraud:",
        summary[
            "precision_fraud"
        ],
    )

    print(
        "predicted_positive_count:",
        summary[
            "predicted_positive_count"
        ],
    )


pair_seconds = (
    time.perf_counter()
    - pair_start
)


print(
    "\n"
    + "=" * 72
)

print(
    "Controlled pair total seconds:",
    round(
        pair_seconds,
        3,
    ),
)

print(
    "\nM5.3 LOGISTIC REGRESSION PAIR: EXECUTED"
)



Starting: M5-LR-LONG-B01
Training window: W_LONG
Training rows: 6,855,270
Training fraud: 9,606

Completed: M5-LR-LONG-B01
fit_seconds: 734.437
prediction_seconds: 0.04
warning_count: 2
f1_fraud: 0.003766478342749529
recall_fraud: 0.0019011406844106464
precision_fraud: 0.2
predicted_positive_count: 10

Starting: M5-LR-SHORT-B01
Training window: W_SHORT
Training rows: 1,721,615
Training fraud: 2,491

Completed: M5-LR-SHORT-B01
fit_seconds: 163.184
prediction_seconds: 0.03
warning_count: 2
f1_fraud: 0.1350210970464135
recall_fraud: 0.07604562737642585
precision_fraud: 0.6015037593984962
predicted_positive_count: 133

Controlled pair total seconds: 897.997

M5.3 LOGISTIC REGRESSION PAIR: EXECUTED


## 15. Warning / convergence gate

M5.3 không được diễn giải model score nếu Logistic Regression chưa hội tụ đủ theo runtime evidence.

Nếu có `ConvergenceWarning`:

`STOP`.

In [13]:

for window_id, output in run_outputs.items():
    summary = output.summary

    print(
        window_id,
        "warnings:",
        summary["warning_count"],
    )

    for warning_message in summary["warnings"]:
        print(
            " -",
            warning_message,
        )

    convergence_warnings = [
        message
        for message
        in summary["warnings"]
        if message.startswith(
            "ConvergenceWarning:"
        )
    ]

    assert not convergence_warnings, (
        f"{window_id}: Logistic Regression "
        "có ConvergenceWarning. "
        "Không được diễn giải metric trước khi "
        "xử lý convergence và rerun controlled pair."
    )

    assert summary["warning_count"] == 0, (
        f"{window_id}: có warning cần review "
        "trước khi khóa baseline."
    )


print(
    "\nM5.3 WARNING / CONVERGENCE GATE: PASS"
)


W_LONG warnings: 2
 - FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
 - ConvergenceWarning: The max_iter was reached which means the coef_ did not converge


AssertionError: W_LONG: Logistic Regression có ConvergenceWarning. Không được diễn giải metric trước khi xử lý convergence và rerun controlled pair.

## 16. Controlled-pair configuration equivalence

In [ ]:

long_summary = run_outputs[
    "W_LONG"
].summary

short_summary = run_outputs[
    "W_SHORT"
].summary


controlled_equal_fields = [
    "runner_version",
    "model_family",
    "model_id",
    "model_config_id",
    "model_class",
    "model_module",
    "model_params",
    "feature_version",
    "preprocessing_version",
    "matrix_schema_version",
    "feature_count",
    "validation_rows",
    "validation_fraud_rows",
    "imbalance_strategy",
    "random_state",
    "threshold_policy",
    "risk_score_kind",
    "final_test_accessed",
]


for field in controlled_equal_fields:
    assert (
        long_summary[field]
        == short_summary[field]
    ), (
        f"Controlled field mismatch: "
        f"{field}"
    )


assert (
    long_summary[
        "training_window_id"
    ]
    == "W_LONG"
)

assert (
    short_summary[
        "training_window_id"
    ]
    == "W_SHORT"
)

assert (
    long_summary[
        "training_rows"
    ]
    == EXPECTED_W_LONG_ROWS
)

assert (
    short_summary[
        "training_rows"
    ]
    == EXPECTED_W_SHORT_ROWS
)

assert (
    long_summary[
        "training_fraud_rows"
    ]
    == EXPECTED_W_LONG_FRAUD
)

assert (
    short_summary[
        "training_fraud_rows"
    ]
    == EXPECTED_W_SHORT_FRAUD
)


print(
    "Controlled equal fields:",
    len(controlled_equal_fields),
)

print(
    "\nOnly intended population factor differs:"
)

print(
    "W_LONG training rows:",
    long_summary[
        "training_rows"
    ],
)

print(
    "W_SHORT training rows:",
    short_summary[
        "training_rows"
    ],
)

print(
    "\nM5.3 CONTROLLED-PAIR CONFIG GATE: PASS"
)


## 17. Probability/risk-score integrity

Logistic Regression hỗ trợ `predict_proba`.

M5.3 phải giữ fraud score và xác nhận nó tương thích với default `predict()` decision rule.

Đây là baseline comparison rule, **không phải final threshold selection**.

In [ ]:

for window_id, output in run_outputs.items():
    summary = output.summary
    score = output.risk_score
    y_pred = output.y_pred

    assert summary[
        "risk_score_available"
    ] is True

    assert (
        summary[
            "risk_score_kind"
        ]
        == "predict_proba"
    )

    assert score is not None

    assert (
        len(score)
        == EXPECTED_VALIDATION_ROWS
    )

    assert np.isfinite(score).all()

    assert np.all(
        score >= 0.0
    )

    assert np.all(
        score <= 1.0
    )

    default_rule_pred = (
        score >= 0.5
    ).astype(
        np.int8
    )

    assert np.array_equal(
        default_rule_pred,
        y_pred,
    ), (
        f"{window_id}: predict() không khớp "
        "probability >= 0.5 default rule."
    )


print(
    "Risk score kind:",
    long_summary[
        "risk_score_kind"
    ],
)

print(
    "Baseline numerical decision point observed:",
    0.5,
)

print(
    "Final threshold status:",
    "OPEN",
)

print(
    "\nM5.3 PROBABILITY / DEFAULT-RULE GATE: PASS"
)


## 18. Metric identity/integrity checks

Cell này chỉ kiểm tra internal arithmetic consistency.

Không chọn winner.

In [ ]:

for window_id, output in run_outputs.items():
    summary = output.summary

    assert summary[
        "validation_rows"
    ] == EXPECTED_VALIDATION_ROWS

    assert summary[
        "validation_fraud_rows"
    ] == EXPECTED_VALIDATION_FRAUD

    assert (
        summary["tn"]
        + summary["fp"]
        + summary["fn"]
        + summary["tp"]
        == EXPECTED_VALIDATION_ROWS
    )

    assert (
        summary[
            "predicted_positive_count"
        ]
        == summary["tp"]
        + summary["fp"]
    )

    expected_predicted_positive_rate = (
        summary[
            "predicted_positive_count"
        ]
        / EXPECTED_VALIDATION_ROWS
    )

    assert np.isclose(
        summary[
            "predicted_positive_rate"
        ],
        expected_predicted_positive_rate,
    )

    for metric_name in [
        "f1_fraud",
        "recall_fraud",
        "precision_fraud",
        "accuracy_reference",
        "predicted_positive_rate",
    ]:
        value = summary[
            metric_name
        ]

        assert np.isfinite(value)
        assert 0.0 <= value <= 1.0

    assert summary[
        "integrity_gate_result"
    ] == "PASS"

    assert summary[
        "errors"
    ] == []

    assert summary[
        "final_test_accessed"
    ] is False


print(
    "M5.3 METRIC / RESULT INTEGRITY GATE: PASS"
)


## 19. Baseline evidence output

Phần này in hai run riêng biệt và delta mô tả.

Không tự gắn nhãn winner.

In [ ]:

def print_run_evidence(
    summary,
):
    print(
        "Experiment:",
        summary["experiment_id"],
    )

    print(
        "Training window:",
        summary[
            "training_window_id"
        ],
    )

    print(
        "Training rows:",
        f"{summary['training_rows']:,}",
    )

    print(
        "Training fraud:",
        f"{summary['training_fraud_rows']:,}",
    )

    print(
        "F1_fraud:",
        summary["f1_fraud"],
    )

    print(
        "Recall_fraud:",
        summary["recall_fraud"],
    )

    print(
        "Precision_fraud:",
        summary[
            "precision_fraud"
        ],
    )

    print(
        "Accuracy reference:",
        summary[
            "accuracy_reference"
        ],
    )

    print(
        "TP / FP / FN / TN:",
        summary["tp"],
        summary["fp"],
        summary["fn"],
        summary["tn"],
    )

    print(
        "Predicted positive count:",
        summary[
            "predicted_positive_count"
        ],
    )

    print(
        "Predicted positive rate:",
        summary[
            "predicted_positive_rate"
        ],
    )

    print(
        "Fit seconds:",
        summary["fit_seconds"],
    )

    print(
        "Prediction seconds:",
        summary[
            "prediction_seconds"
        ],
    )


print("=" * 72)
print_run_evidence(
    long_summary
)

print("\n" + "=" * 72)
print_run_evidence(
    short_summary
)


print("\n" + "=" * 72)
print("Descriptive deltas: W_LONG - W_SHORT")

for metric_name in [
    "f1_fraud",
    "recall_fraud",
    "precision_fraud",
    "accuracy_reference",
    "predicted_positive_rate",
]:
    delta = (
        long_summary[
            metric_name
        ]
        - short_summary[
            metric_name
        ]
    )

    print(
        metric_name,
        delta,
    )


print(
    "\nTraining-window winner:",
    "OPEN — M5.3 DOES NOT SELECT",
)


## 20. Persist LR-LONG / LR-SHORT prediction evidence

In [ ]:

saved_outputs = {}


for window_id, output in run_outputs.items():
    saved_outputs[
        window_id
    ] = save_experiment_output(
        output,
        OUTPUT_DIR,
    )


for window_id, paths in saved_outputs.items():
    print(
        "\n",
        window_id,
        sep="",
    )

    for key, path in paths.items():
        print(
            key,
            "→",
            path,
        )

        if path is not None:
            assert path.exists()
            assert path.stat().st_size > 0


print(
    "\nM5.3 PREDICTION ARTIFACT PERSISTENCE: PASS"
)


## 21. Saved-artifact round-trip verification

In [ ]:

for window_id, output in run_outputs.items():
    paths = saved_outputs[
        window_id
    ]

    reloaded_pred = np.load(
        paths[
            "prediction_path"
        ],
        allow_pickle=False,
    )

    assert np.array_equal(
        reloaded_pred,
        output.y_pred,
    )

    assert (
        reloaded_pred.dtype
        == np.int8
    )

    assert (
        len(reloaded_pred)
        == EXPECTED_VALIDATION_ROWS
    )


    assert (
        paths[
            "risk_score_path"
        ]
        is not None
    )

    reloaded_score = np.load(
        paths[
            "risk_score_path"
        ],
        allow_pickle=False,
    )

    assert np.array_equal(
        reloaded_score,
        output.risk_score,
    )

    assert (
        reloaded_score.dtype
        == np.float32
    )

    assert (
        len(reloaded_score)
        == EXPECTED_VALIDATION_ROWS
    )


    with open(
        paths[
            "summary_path"
        ],
        "r",
        encoding="utf-8",
    ) as file:
        reloaded_summary = json.load(
            file
        )

    assert (
        reloaded_summary[
            "experiment_id"
        ]
        == output.summary[
            "experiment_id"
        ]
    )

    assert (
        reloaded_summary[
            "model_config_id"
        ]
        == LR_CONFIG_ID
    )

    assert (
        reloaded_summary[
            "final_test_accessed"
        ]
        is False
    )


print(
    "M5.3 SAVED ARTIFACT ROUND-TRIP GATE: PASS"
)


## 22. Persist controlled-pair manifest

Manifest này ghi lại pair identity + config fingerprint + artifact names.

Không khóa training-window winner.

In [ ]:

pair_manifest = {
    "m5_substep": "M5.3",
    "model_family": LR_MODEL_FAMILY,
    "model_id": LR_MODEL_ID,
    "model_config_id": LR_CONFIG_ID,
    "config_fingerprint_sha256": config_fingerprint,
    "runner_version": M5_RUNNER_VERSION,
    "feature_version": FEATURE_VERSION,
    "preprocessing_version": PREPROCESSING_VERSION,
    "matrix_schema_version": MATRIX_SCHEMA_VERSION,
    "imbalance_strategy": IMBALANCE_STRATEGY,
    "random_state": DEFAULT_RANDOM_STATE,
    "threshold_policy": BASELINE_THRESHOLD_POLICY,
    "final_threshold_status": "OPEN",
    "final_test_accessed": False,
    "training_window_winner": "OPEN",
    "runs": {
        "W_LONG": {
            "experiment_id":
                long_summary[
                    "experiment_id"
                ],
            "summary_file":
                saved_outputs[
                    "W_LONG"
                ][
                    "summary_path"
                ].name,
            "prediction_file":
                saved_outputs[
                    "W_LONG"
                ][
                    "prediction_path"
                ].name,
            "risk_score_file":
                saved_outputs[
                    "W_LONG"
                ][
                    "risk_score_path"
                ].name,
        },
        "W_SHORT": {
            "experiment_id":
                short_summary[
                    "experiment_id"
                ],
            "summary_file":
                saved_outputs[
                    "W_SHORT"
                ][
                    "summary_path"
                ].name,
            "prediction_file":
                saved_outputs[
                    "W_SHORT"
                ][
                    "prediction_path"
                ].name,
            "risk_score_file":
                saved_outputs[
                    "W_SHORT"
                ][
                    "risk_score_path"
                ].name,
        },
    },
}


PAIR_MANIFEST_PATH = (
    OUTPUT_DIR
    / "m5_03_lr_pair_manifest.json"
)


with open(
    PAIR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        pair_manifest,
        file,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


assert PAIR_MANIFEST_PATH.exists()

with open(
    PAIR_MANIFEST_PATH,
    "r",
    encoding="utf-8",
) as file:
    reloaded_pair_manifest = json.load(
        file
    )


assert (
    reloaded_pair_manifest[
        "config_fingerprint_sha256"
    ]
    == config_fingerprint
)

assert (
    reloaded_pair_manifest[
        "training_window_winner"
    ]
    == "OPEN"
)

assert (
    reloaded_pair_manifest[
        "final_test_accessed"
    ]
    is False
)


print("Pair manifest:")
print(PAIR_MANIFEST_PATH)

print(
    "\nM5.3 PAIR MANIFEST GATE: PASS"
)


## 23. FINAL TEST isolation regression gate

In [ ]:

assert manifest["final_test_used"] is False
assert m5_02_contract["final_test_access_allowed"] is False

assert set(MODELING_INPUTS) == {
    "W_LONG",
    "W_SHORT",
}

assert long_summary[
    "final_test_accessed"
] is False

assert short_summary[
    "final_test_accessed"
] is False

assert pair_manifest[
    "final_test_accessed"
] is False


for path in OUTPUT_DIR.iterdir():
    assert (
        "final_test"
        not in path.name.lower()
    ), (
        "Phát hiện tên artifact liên quan FINAL TEST: "
        f"{path.name}"
    )


print(
    "M5.3 FINAL TEST ISOLATION GATE: PASS"
)


## 24. M5.3 overall technical gate

In [ ]:

m5_03_gates = {
    "G01_MANIFEST_CONTRACT": True,
    "G02_INPUT_INTEGRITY": True,
    "G03_LR_CONFIG_DEFINED": True,
    "G04_PRE_RESULT_CONFIG_LOCK": True,
    "G05_MODELING_INPUTS": True,
    "G06_PAIR_EXECUTED": True,
    "G07_WARNING_CONVERGENCE": True,
    "G08_CONTROLLED_PAIR_CONFIG": True,
    "G09_PROBABILITY_DEFAULT_RULE": True,
    "G10_METRIC_RESULT_INTEGRITY": True,
    "G11_PREDICTION_PERSISTENCE": True,
    "G12_ARTIFACT_ROUND_TRIP": True,
    "G13_PAIR_MANIFEST": True,
    "G14_FINAL_TEST_ISOLATION": True,
}


for gate_name, gate_value in m5_03_gates.items():
    print(
        gate_name,
        "→",
        (
            "PASS"
            if gate_value
            else "FAIL"
        ),
    )


assert all(
    m5_03_gates.values()
)


print(
    "\nM5.3 OVERALL TECHNICAL GATE: PASS"
)


# 25. Runtime review registry

Sau khi Run All, cần review các finding sau.

## M5.3-F01 — Exact baseline configuration

Cần xác nhận config lock được tạo trước model runs và cùng fingerprint áp dụng cho pair.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F02 — Logistic Regression fit integrity

Cần xác nhận:

- LR-LONG fit hoàn tất;
- LR-SHORT fit hoàn tất;
- không ConvergenceWarning;
- không warning/error khác cần xử lý.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F03 — Controlled training-window pair

Cần xác nhận mọi field ngoài training population đều được kiểm soát.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F04 — Baseline fraud-class behavior

Cần đọc:

- F1_fraud;
- Recall_fraud;
- Precision_fraud;
- TP/FP/FN/TN;
- predicted-positive count/rate.

Chỉ mô tả baseline evidence.

Không chọn final winner ở M5.3.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F05 — Probability output

Cần xác nhận:

- score finite;
- trong [0, 1];
- default prediction khớp score >= 0.5;
- risk-score artifacts được lưu.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F06 — Persistence / reproducibility

Cần xác nhận:

- config lock;
- summary;
- predictions;
- risk scores;
- pair manifest;
- round-trip.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F07 — FINAL TEST isolation

Cần xác nhận final-test access vẫn bằng 0.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

---

## M5.3-F08 — Handoff

Cần quyết định M5.3 có đủ evidence để handoff sang M5.4 hay không.

Status:

`OPEN — CHỜ RUNTIME REVIEW`

# 26. Decision Log — trước runtime review

## M5.3-D01 — Logistic Regression baseline config

Decision:

`LR-B01-SAGA-L2-C1`

```text
penalty = l2
C = 1.0
solver = saga
max_iter = 200
tol = 1e-4
fit_intercept = True
class_weight = None
random_state = 42
```

Status:

`LOCKED BEFORE RESULT`

---

## M5.3-D02 — Imbalance strategy

Decision:

`NONE`

Status:

`INHERITED — LOCKED`

---

## M5.3-D03 — Training-window pair

Decision:

Run both:

`LR-LONG`

`LR-SHORT`

với cùng exact config.

Status:

`LOCKED`

---

## M5.3-D04 — Threshold policy

Decision:

`DEFAULT_MODEL_DECISION_RULE`

Đối với Logistic Regression baseline, notebook kiểm tra `predict()` khớp `P(fraud) >= 0.5`.

Đây không phải final threshold.

Status:

`LOCKED FOR BASELINE`

Final threshold:

`OPEN`

---

## M5.3-D05 — Winner policy

Decision:

M5.3 không tự khóa W_LONG/W_SHORT winner.

Status:

`OPEN — REQUIRES LATER EVALUATION/SELECTION`

---

## M5.3-D06 — Systematic tuning

Decision:

Không thực hiện trong M5.3.

Status:

`DEFERRED TO M7`

---

## M5.3-D07 — FINAL TEST

Decision:

No access.

Status:

`INHERITED — LOCKED`

---

## M5.3-D08 — M5.4 handoff

Status:

`OPEN — CHỜ RUNTIME REVIEW`

# 27. Kết luận trước runtime

Notebook M5.3 đã khóa exact Logistic Regression baseline config trước model result và triển khai controlled pair:

`M5-LR-LONG-B01`

`M5-LR-SHORT-B01`

Trước Run All:

```text
M5.3:
NOT YET VERIFIED

LR baseline config:
LOCKED BEFORE RESULT

Imbalance:
NONE

W_LONG / W_SHORT:
READY TO RUN

Metric contract:
INHERITED FROM M5.2

Risk score:
REQUIRED

Threshold:
DEFAULT MODEL DECISION RULE
NOT FINAL

Systematic tuning:
NO

FINAL TEST:
PROTECTED

Training-window winner:
OPEN

Next:
Run All → runtime review
```

Chỉ sau runtime review mới được kết luận:

`M5.3 — PASS`

và handoff:

`READY FOR M5.4`.